In [1]:
import random
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data.sampler import SubsetRandomSampler

from torchsummary import summary

import torchvision.transforms as T
from torchvision.datasets import MNIST

from sklearn.manifold import TSNE

from matplotlib import pyplot as plt

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-wheg2c3t because the default path (/home/20093355/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
def make_fig(z, targets, epoch):

    model = TSNE()
    transformed = model.fit_transform(z)

    plt.scatter(transformed[:, 0], transformed[:, 1], c=targets, alpha=.4, s=3**2)

    plt.savefig(fname=f'{epoch}.png', format='png')

In [3]:
LR = 0.0001
EPOCH = 200
BATCH_SIZE = 64

LATENT_DIM = 128

In [4]:
class SubsetSequentialSampler(torch.utils.data.Sampler):
    def __init__(self, indices):
        self.indices = indices

    def __iter__(self):
        return (self.indices[i] for i in range(len(self.indices)))
    
    def __len__(self):
        return len(self.indices)

train = MNIST('./', train=True, download=True,
              transform=T.Compose([T.ToTensor(),
                                   T.Normalize((0.5,), (0.5,)),])
             )
test = MNIST('./', train=False, download=True, transform=T.ToTensor())

train_loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=BATCH_SIZE)

In [5]:
class Residual(nn.Module):
    def __init__(self, channel):
        super(Residual, self).__init__()

        self.conv1 = nn.Conv2d(channel, channel, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(channel, channel, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(channel)
        self.bn2 = nn.BatchNorm2d(channel)

        self.leaky = nn.LeakyReLU(inplace=False)

    def forward(self, x):
        out = self.leaky(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + x
        return self.leaky(out)

    
class Pooling(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(Pooling, self).__init__()

        self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(out_channel, out_channel, kernel_size=3, stride=1, padding=1)
        
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.bn2 = nn.BatchNorm2d(out_channel)

        self.leaky = nn.LeakyReLU(inplace=False)

    def forward(self, x):
        out_ = self.leaky(self.bn1(self.conv1(x)))
        return out_

In [6]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.block_1 = nn.Sequential(Residual(1), Pooling(1, 8))  # 14
        self.block_2 = nn.Sequential(Residual(8), Pooling(8, 16))  # 7
        
        self.fc = nn.Linear(16, 1)

        self.GAP = nn.AdaptiveAvgPool2d(1)

        self.leaky = nn.LeakyReLU(inplace=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.block_1(x)
        out = self.block_2(out)

        out = self.GAP(out)

        out = out.view(out.size(0), -1)
        
        out = self.fc(out)
        
        return self.sigmoid(out)

In [7]:
class Transpose(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(Transpose, self).__init__()

        self.conv_trans = nn.ConvTranspose2d(in_channels=in_channel,
                                             out_channels=out_channel,
                                             kernel_size=4, stride=2, padding=1)
        self.conv1 = nn.Conv2d(out_channel, out_channel, kernel_size=3, stride=1, padding=1)
        
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.bn2 = nn.BatchNorm2d(out_channel)
        
        self.leaky = nn.LeakyReLU(inplace=False)

    def forward(self, x):
        out_ = self.leaky(self.bn1(self.conv_trans(x)))
        out = self.bn2(self.conv1(out_))
        out = out + out_
        return out

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        
        self.fc = nn.Linear(LATENT_DIM, 7*7*16)
        
        self.bn = nn.BatchNorm1d(7*7*16)

        self.transpose_1 = Transpose(16, 8)
        self.transpose_2 = Transpose(8, 1)

        self.leaky = nn.LeakyReLU(inplace=False)
        self.tanh = nn.Tanh()

    def forward(self, x):
        out = self.leaky(self.bn(self.fc(x)))

        out = out.view(out.size(0), 16, 7, 7)

        out = self.leaky(self.transpose_1(out))
        out = self.transpose_2(out)

        return self.tanh(out)

In [8]:
disc = Discriminator().cuda()
gen = Generator().cuda()

g_optim = torch.optim.Adam(gen.parameters(), lr=2e-4)
d_optim = torch.optim.Adam(disc.parameters(), lr=5e-5)

criterion = nn.BCELoss().cuda()

In [9]:
def train_discriminator(criterion, optimizer, real_data, fake_data):
    n = real_data.size(0)

    optimizer.zero_grad()

    prediction_real = disc(real_data)
    loss_real = criterion(prediction_real, torch.ones(n, 1).cuda())
    loss_real.backward()

    prediction_fake = disc(fake_data)
    loss_fake = criterion(prediction_fake, torch.zeros(n, 1).cuda())
    loss_fake.backward()

    optimizer.step()

    return loss_real + loss_fake

def train_generator(criterion, optimizer, fake_data):
    n = fake_data.size(0)
    optimizer.zero_grad()
    
    prediction = disc(fake_data)
    loss = criterion(prediction, torch.ones(n, 1).cuda())
    
    loss.backward()
    optimizer.step()
    
    return loss


In [10]:
for epoch in range(EPOCH):
    gen.train()
    disc.train()
    for img, target in tqdm(train_loader, leave=False, total=len(train_loader)):
        img = img.cuda()
        target = target.cuda()
        
        n = len(img)
        
        noise = torch.randn(n, LATENT_DIM).cuda()
        fake_data = gen(noise).detach()
        disc_loss = train_discriminator(criterion, d_optim, img, fake_data)
        
        noise = torch.randn(n, LATENT_DIM).cuda()
        fake_data = gen(noise)
        gen_loss = train_generator(criterion, g_optim, fake_data)
        
    if not epoch % 10:
        print(disc_loss, gen_loss)
        display(T.functional.to_pil_image(gen(torch.randn(2, LATENT_DIM).cuda())[0]))

tensor(1.2904, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.7560, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(1.1535, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.8871, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(1.1684, device='cuda:0', grad_fn=<AddBackward0>) tensor(1.1517, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(1.2473, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.8727, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(0.9976, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.8887, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


...

tensor(0.6947, device='cuda:0', grad_fn=<AddBackward0>) tensor(1.4357, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(0.7402, device='cuda:0', grad_fn=<AddBackward0>) tensor(1.5333, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(0.8213, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.9748, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(1.0473, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.8608, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


tensor(0.3776, device='cuda:0', grad_fn=<AddBackward0>) tensor(1.7427, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>)


In [11]:
T.functional.to_pil_image(gen(torch.randn(2, LATENT_DIM).cuda())[0])